In [ ]:
import os
from Bio import motifs

jaspar_file = "../../data/Motifs/JASPAR2026_CORE_vertebrates_non-redundant_pfms_jaspar.txt"
out_dir = "../../results/03_accessbpnet/13_prepare_pwm"
os.makedirs(out_dir, exist_ok=True)
out_tsv = os.path.join(out_dir, "motif_to_pwm.tsv")

# Representative TFs for comparing corrected vs uncorrected footprints (shared by K562 + HepG2)
representative_tfs = [
    # strong, canonical footprints (ChIP-seq ground truth in both cell lines)
    "CTCF", "MAX", "USF1", "SP1", "NRF1", "YY1", "E2F1", "ELF1", "GABPA",
    # K562 lineage
    "GATA1", "TAL1", "NFE2", "RUNX1", "MYB",
    # HepG2 lineage
    "HNF4A", "FOXA2", "CEBPA", "HNF1A",
    # AP-1 / others
    "JUN", "FOSL2::JUND",
]

tf_set = {t.upper() for t in representative_tfs}

written, seen = [], set()
with open(jaspar_file) as f, open(out_tsv, "w") as out:
    for m in motifs.parse(f, "jaspar"):
        # split dimers (e.g. FOSL2::JUND); match if the full name or any partner is whitelisted
        name_u = m.name.upper()
        partners = name_u.split("::")
        hit = name_u in tf_set or any(p in tf_set for p in partners)
        if not hit:
            continue
        # keep one motif per TF name
        if name_u in seen:
            continue
        seen.add(name_u)

        # highest-probability base at each position (no IUPAC degenerate codes)
        consensus = str(m.consensus)
        out.write(f"{m.name}_{m.matrix_id}\t{consensus}\n")
        written.append(m.name)

print(f"wrote {len(written)} motifs -> {out_tsv}")
print("  ", ", ".join(sorted(written)))

# report which requested TFs were not found in JASPAR
found = {n.upper() for n in written}
found_flat = set()
for n in found:
    found_flat.update(n.split("::"))
missing = [t for t in representative_tfs if t.upper() not in found and t.upper() not in found_flat]
if missing:
    print("  not found:", ", ".join(missing))